# 3) The DC approximation — Exercises

**Goals:** write a DC power flow from a line table, estimate losses from DC flows, use DC angles to start an AC solver, find an overload that DC cannot see, and test whether DC ranks line outages correctly.

The setup cell holds the lecture's Croatian network (`croatia_pandapower` with switches for the Zagreb voltage set point, line ratings and an outage; `croatia_pypsa`), and lesson 1's `case14_data` and `newton_raphson`. Power in MW, 100 MVA base.

**Submit** your notebook for review before opening the solution in `solutions/`.

In [ ]:
import logging
import warnings

import numpy as np
import pandas as pd
import pandapower as pp
import pandapower.networks as pn
import pypsa

warnings.filterwarnings("ignore")
for name in ("pandapower", "pypsa", "linopy"):
    logging.getLogger(name).setLevel(logging.ERROR)

# ---- the lecture's stylised Croatian 400 kV network, with a few switches for the exercises ----
BUSES = ["Zagreb", "Rijeka", "Split", "Osijek", "Velebit"]
LINES = pd.DataFrame([("Zagreb", "Rijeka", 130), ("Rijeka", "Velebit", 110), ("Velebit", "Split", 160),
                      ("Zagreb", "Split", 320), ("Zagreb", "Osijek", 250), ("Osijek", "Split", 330)],
                     columns=["bus0", "bus1", "km"])
R_KM, X_KM, C_KM = 0.028, 0.32, 11.5
LOAD = {"Zagreb": 900, "Rijeka": 300, "Split": 450, "Osijek": 250}
GEN = {"Rijeka": 500, "Split": 600, "Velebit": 300}


def croatia_pandapower(v_zagreb=1.0, rating_ka=None, out_of_service=None):
    """The lecture's network. rating_ka: {line position: kA}; out_of_service: line position to switch off."""
    net = pp.create_empty_network(sn_mva=100)
    bus = {name: pp.create_bus(net, 400, name=name) for name in BUSES}
    pp.create_ext_grid(net, bus["Zagreb"], vm_pu=v_zagreb)
    for name, p in GEN.items():
        pp.create_gen(net, bus[name], p_mw=p, vm_pu=1.0)
    for name, p in LOAD.items():
        pp.create_load(net, bus[name], p_mw=p, q_mvar=0.2 * p)
    for i, line in LINES.iterrows():
        pp.create_line_from_parameters(net, bus[line.bus0], bus[line.bus1], line.km, R_KM, X_KM, C_KM,
                                       max_i_ka=(rating_ka or {}).get(i, 2.0), in_service=(i != out_of_service))
    return net


def croatia_pypsa():
    n = pypsa.Network()
    n.add("Bus", BUSES, v_nom=400)
    n.add("Generator", "slack", bus="Zagreb", control="Slack")
    for name, p in GEN.items():
        n.add("Generator", f"{name} gen", bus=name, p_set=p, control="PV")
    for name, p in LOAD.items():
        n.add("Load", f"{name} load", bus=name, p_set=p, q_set=0.2 * p)
    for _, line in LINES.iterrows():
        n.add("Line", f"{line.bus0}-{line.bus1}", bus0=line.bus0, bus1=line.bus1, r=R_KM * line.km, x=X_KM * line.km,
              b=2 * np.pi * 50 * C_KM * 1e-9 * line.km, s_nom=np.sqrt(3) * 400 * 2.0)
    return n


# ---- lesson 1's Newton-Raphson and the IEEE 14-bus internal data ----
def case14_data(load_scale=1.0):
    net = pn.case14()
    net.load[["p_mw", "q_mvar"]] *= load_scale
    net.gen.p_mw *= load_scale
    pp.runpp(net, init="flat", numba=False, tolerance_mva=1e-10)
    internal, order = net._ppc["internal"], net._pd2ppc_lookups["bus"]
    v_set = np.ones(len(internal["Sbus"]))
    for table in (net.gen, net.ext_grid):
        v_set[order[table.bus.to_numpy()]] = table.vm_pu.to_numpy()
    return net, {"Y": internal["Ybus"].toarray(), "S": internal["Sbus"].copy(), "v_set": v_set, "pv": internal["pv"],
                 "pq": internal["pq"], "order": order}


def newton_raphson(Y, S, V0, pv, pq, tol=1e-10, max_iter=30):
    """Lesson 1's solver. Returns V and the number of iterations."""
    V = V0.astype(complex).copy()
    pvpq = np.r_[pv, pq]
    for it in range(max_iter + 1):
        I = Y @ V
        m = V * np.conj(I) - S
        F = np.r_[m[pvpq].real, m[pq].imag]
        if np.abs(F).max() < tol:
            return V, it
        Vn = V / np.abs(V)
        dS_dVa = 1j * np.diag(V) @ np.conj(np.diag(I) - Y @ np.diag(V))
        dS_dVm = np.diag(V) @ np.conj(Y @ np.diag(Vn)) + np.conj(np.diag(I)) @ np.diag(Vn)
        J = np.block([[dS_dVa[np.ix_(pvpq, pvpq)].real, dS_dVm[np.ix_(pvpq, pq)].real],
                      [dS_dVa[np.ix_(pq, pvpq)].imag, dS_dVm[np.ix_(pq, pq)].imag]])
        dx = np.linalg.solve(J, -F)
        Va, Vm = np.angle(V), np.abs(V)
        Va[pvpq] += dx[:len(pvpq)]
        Vm[pq] += dx[len(pvpq):]
        V = Vm * np.exp(1j * Va)
    raise RuntimeError("no convergence")

### Warm-ups

**1. A DC power flow from the line table**

Write `dc_flows()` for the Croatian network directly from `LINES`, `LOAD` and `GEN`: convert each line's reactance to per unit (x_pu = x_Ω / (400² / 100)), build B, fix the slack angle at Zagreb to zero, solve for the other angles and return the line flows in MW. Check them against PyPSA's `lpf()`.

In [ ]:
def dc_flows() -> pd.Series:
    """DC line flows (MW) of the Croatian network, from bus0 to bus1, indexed like PyPSA's lines."""
    ...  # YOUR CODE HERE

In [ ]:
own = dc_flows()
n = croatia_pypsa()
n.lpf()
assert np.abs(own.to_numpy() - n.lines_t.p0.iloc[0].to_numpy()).max() < 1e-6
own.round(1)

**2. Losses from DC flows**

DC is lossless, but its flows can estimate the losses: on a line, losses ≈ r · P² / V² (with V = 1 p.u.). Write `dc_loss_estimate(flows_mw)` returning the total estimated losses (MW) for the Croatian network, and compare with the AC losses from pandapower.

In [ ]:
def dc_loss_estimate(flows_mw: pd.Series) -> float:
    """Sum over lines of r_pu * P_pu^2, in MW (100 MVA base, 400 kV)."""
    ...  # YOUR CODE HERE

In [ ]:
estimate = dc_loss_estimate(own)
net = croatia_pandapower()
pp.runpp(net, numba=False)
actual = net.res_line.pl_mw.sum()
assert abs(estimate - actual) / actual < 0.15, "within 15 % of the AC losses"
print(f"DC-based estimate {estimate:.2f} MW, AC losses {actual:.2f} MW ({100 * (estimate / actual - 1):+.1f} %)")

### Core

**3. DC angles as a starting point**

Newton–Raphson from a flat start begins at all angles zero. Start it instead from the **DC angles** (and the set-point voltages at slack and PV buses, 1 p.u. elsewhere). For the IEEE 14-bus system at load scales [1.0, 2.0, 3.0, 4.0] (generators scaled too), compare the iterations from a flat start and from the DC start. Write `dc_start(load_scale)`; use pandapower's `rundcpp` for the DC angles and map them into the internal order with `order`.

In [ ]:
def dc_start(load_scale: float) -> dict:
    """Newton iterations from a flat start and from the DC solution."""
    ...  # YOUR CODE HERE

In [ ]:
starts = pd.DataFrame({s: dc_start(s) for s in [1.0, 2.0, 3.0, 4.0]}).T.rename_axis("load scale")
assert (starts["DC start"] <= starts["flat start"]).all()
starts

**4. An overload DC cannot see**

Give the Zagreb–Rijeka line (position 0) a thermal rating of 0.55 kA. Sweep the Zagreb voltage set point over [1.00, 1.01, …, 1.05] and report, for each, the line's active and reactive flow (AC), its loading by DC (|P_DC| / (√3 · 400 kV · 0.55 kA)) and its loading by AC (pandapower's `loading_percent`). From which set point does AC report an overload that DC misses? Write `hidden_overload(setpoints)`.

In [ ]:
def hidden_overload(setpoints) -> pd.DataFrame:
    """Zagreb-Rijeka flows and loadings (DC and AC) for each Zagreb voltage set point."""
    ...  # YOUR CODE HERE

In [ ]:
sweep = hidden_overload(np.round(np.arange(1.00, 1.051, 0.01), 2))
assert sweep["DC_loading_%"].max() < 100 and sweep["AC_loading_%"].max() > 100
missed = sweep.index[(sweep["AC_loading_%"] > 100) & (sweep["DC_loading_%"] < 100)]
print("overloads DC misses at set points:", list(missed))
sweep.round(1)

**5. Interpretation**

How good was the loss estimate, and why does it work although DC is lossless? How much does a DC start save Newton–Raphson, and when? Why can DC not see the overload in task 4, and which kind of study must therefore use AC?

*Your answer here.*

### Challenge

**6. Does DC rank the outages right?**

Security screening asks: which single line outage stresses the network most? For each of the six lines of the Croatian network, take it out of service and compute the largest line flow (MW) with AC and with DC. Rank the outages by each method and compare. Write `n_minus_1()`.

In [ ]:
def n_minus_1() -> pd.DataFrame:
    """Largest line flow after each single line outage, AC and DC, with ranks."""
    ...  # YOUR CODE HERE

In [ ]:
screen = n_minus_1()
assert len(screen) == 6
assert (screen.DC_max_MW - screen.AC_max_MW).abs().max() < 20
screen.round(1)

**7. Interpretation**

Did DC rank the outages in the same order as AC? What does that suggest for how screening studies combine DC and AC power flows?

*Your answer here.*